# gatenet confidence head on Colab -- TAG `colab-conf`

Trains the per-crop trust/reject head (`gatenet_conf.py`) on top of the **frozen**
production regressor `gatenet_runs/colab-v3-nw/best.pt`. The regressor path stays
bit-identical (asserted in cell 9); only a 328k-param head learns. Design rationale is
the `gatenet_conf.py` module docstring; the short version:

* **class 1** = merged v3 labels (unsure excluded);
* **class 0** = explicit negatives ONLY -- behind-gate (527 -> capped), no-orange
  phantoms (437 -> capped, degenerate boxes excluded), mined VQ2 decoration FPs on
  Claire's hand-labelled frames, verified-empty randoms. LABEL_POLICY.md forbids
  "unlabelled region = background", and cell 4 re-asserts that on the uploaded files.
* second output: predicted log10 regressor error -- the "clipped/degraded" signal as a
  continuous quantity instead of a third class.

## FROZEN accept criteria (written before any number below existed)

**Deploy the head only if it rejects >= 80% of decoration false positives at <= 2%
true-gate loss on the clean eval set.** Catastrophe catch is reported against the PnP
residual baseline (84% of >10 px at 12.6% flags) as a complement, not a gate.

## Upload list -- Drive `MyDrive/vqual2/` (run `mkcolab.py` locally first)

```
MyDrive/vqual2/
  perception/                  <- from colab_upload/perception/ (replace on collision)
    gatenet.py  packcrops.py  autolabel.py  label.py       (autolabel/label already there)
    gatenet_conf.py            <- NEW
    labels_merged_v3.json      val_clean_keys.json
    labelfix_negatives_v3.json autolabels_vq1_negatives.json   <- NEW
    conf_negatives_vq2.json    labels_gates_all.json           <- NEW
    vq2_label_index.json       <- NEW (copy of vq2_label/index.json)
  vq1_frames.zip               <- already there
  vq2_frames.zip               <- already there
  conf_frames_vq1.zip          <- NEW, 1.6 MB (the 173 phantom-negative frames)
  gatenet_runs/colab-v3-nw/best.pt   <- already there (the frozen trunk)
```

Then Runtime -> Run all. If it disconnects: reconnect, set `RESUME = True` in the train
cell, Run all again.

In [ ]:
import os, subprocess, sys, time, shutil, json

print("--- GPU " + "-" * 60)
try:
    print(subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
         "--format=csv,noheader"], text=True).strip())
except Exception as e:
    print("no nvidia-smi:", e)

import torch
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
print("vCPU:", os.cpu_count())
t, u, f = shutil.disk_usage("/content")
print(f"/content: total {t/2**30:.0f} GiB, free {f/2**30:.0f} GiB")
assert f > 4 * 2**30, "not enough local disk for frames + checkpoints"

## 2. Mount Drive and check every input BEFORE spending time

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE       = "/content/drive/MyDrive/vqual2"    # <- the one path to change
CODE_DRIVE  = f"{DRIVE}/perception"
VQ1_ZIP     = f"{DRIVE}/vq1_frames.zip"
VQ2_ZIP     = f"{DRIVE}/vq2_frames.zip"
CONF_ZIP    = f"{DRIVE}/conf_frames_vq1.zip"     # 173 phantom-negative frames
RUNS_DRIVE  = f"{DRIVE}/gatenet_runs"
TRUNK_DRIVE = f"{RUNS_DRIVE}/colab-v3-nw/best.pt"   # PRODUCTION regressor, frozen

CODE_LOCAL  = "/content/code"
SESS_LOCAL  = "/content/sessions"

need_files = [VQ1_ZIP, VQ2_ZIP, CONF_ZIP, TRUNK_DRIVE] + [
    f"{CODE_DRIVE}/{n}" for n in (
        "gatenet.py", "autolabel.py", "gatenet_conf.py",
        "labels_merged_v3.json", "val_clean_keys.json",
        "labelfix_negatives_v3.json", "autolabels_vq1_negatives.json",
        "conf_negatives_vq2.json", "labels_gates_all.json", "vq2_label_index.json")]
missing = [p for p in need_files if not os.path.isfile(p)]
assert not missing, "missing on Drive (see the upload list at the top):\n  " + \
    "\n  ".join(missing)
os.makedirs(f"{RUNS_DRIVE}/colab-conf", exist_ok=True)
print("Drive OK --", len(need_files), "inputs present")

## 3. Code + frames onto local disk (never train against the FUSE mount)

In [ ]:
import zipfile

os.makedirs(f"{CODE_LOCAL}/pilot", exist_ok=True)
shutil.rmtree(f"{CODE_LOCAL}/pilot/perception", ignore_errors=True)
shutil.copytree(CODE_DRIVE, f"{CODE_LOCAL}/pilot/perception")
# gatenet_conf expects vq2_label/index.json relative to itself
os.makedirs(f"{CODE_LOCAL}/pilot/perception/vq2_label", exist_ok=True)
shutil.copyfile(f"{CODE_LOCAL}/pilot/perception/vq2_label_index.json",
                f"{CODE_LOCAL}/pilot/perception/vq2_label/index.json")

if not os.path.isdir(SESS_LOCAL):
    os.makedirs(SESS_LOCAL, exist_ok=True)
    for zp in (VQ1_ZIP, VQ2_ZIP, CONF_ZIP):
        t0 = time.time()
        with zipfile.ZipFile(zp) as z:
            n = len(z.namelist())
            z.extractall(SESS_LOCAL)
        print(f"unzipped {n} frames from {os.path.basename(zp)} in {time.time()-t0:.0f} s")
else:
    print("frames already unzipped")

link = f"{CODE_LOCAL}/pilot/sessions"
if not os.path.islink(link) and not os.path.isdir(link):
    os.symlink(SESS_LOCAL, link)

TRUNK_LOCAL = "/content/trunk_best.pt"
shutil.copyfile(TRUNK_DRIVE, TRUNK_LOCAL)
print("trunk:", TRUNK_LOCAL, f"{os.path.getsize(TRUNK_LOCAL)/1e6:.1f} MB")

## 4. Build the index; re-assert the label-policy invariants on the UPLOADED files

`check_invariants` re-derives the policy guarantees from the raw jsons rather than
trusting the local machine that mined them: every negative from an explicit source, no
hand-frame unlabelled region ever used as background, degenerate boxes excluded. A
stale upload fails here, in seconds, not after training.

In [ ]:
sys.path.insert(0, f"{CODE_LOCAL}/pilot/perception")
import numpy as np
import gatenet as G
import gatenet_conf as C

G.LABELS = f"{CODE_LOCAL}/pilot/perception/labels_merged_v3.json"
items, counts = C.build_index()
for k in sorted(k for k in counts if isinstance(k, tuple)):
    print(f"  cls={k[0]}  {k[1]:14s} {counts[k]:6d}")
print(f"negatives before dedupe {counts['neg_before_dedupe']}, "
      f"unsure excluded {counts['unsure_excluded']}, "
      f"degenerate excluded {counts['degenerate_excluded']}")

miss = [d['key'] for d in items if not os.path.exists(d['path'])]
assert not miss, f"{len(miss)} frames missing after unzip, e.g. {miss[:5]}"
C.check_invariants(items)

tr_items, va_items = C.conf_split(items)
print(f"split: train {len(tr_items)} / val {len(va_items)}  "
      f"(val negs {sum(1 for d in va_items if not d['cls'])}, "
      f"val deco {sum(1 for d in va_items if d['reason']=='deco')})")

### 4b. Sanity: the crops are what the counts claim

In [ ]:
import cv2
from matplotlib import pyplot as plt

ds = C.ConfCrops(va_items, train=False)
picks = []
for want in ('gate', 'deco', 'behind-gate', 'no-orange', 'empty-random'):
    ks = [k for k, d in enumerate(va_items) if d['reason'] == want]
    picks += list(np.array(ks)[np.linspace(0, len(ks) - 1, min(4, len(ks))).astype(int)]) if ks else []
fig, ax = plt.subplots((len(picks) + 3) // 4, 4, figsize=(14, 3.5 * ((len(picks) + 3) // 4)))
for a in np.ravel(ax):
    a.axis("off")
for a, k in zip(np.ravel(ax), picks):
    x, cls, tgt, geo, _ = ds[int(k)]
    img = ((x * 0.25 + 0.45) * 255).clamp(0, 255).byte().numpy().transpose(1, 2, 0)
    img = np.ascontiguousarray(img[:, :, ::-1])
    d = va_items[int(k)]
    if d['cls'] == 1:
        q = (tgt.numpy().reshape(4, 2) + 1.0) * (G.RES / 2.0)
        cv2.polylines(img, [q.astype(np.int32).reshape(-1, 1, 2)], True, (0, 255, 0), 1)
    a.imshow(img)
    a.set_title(f"{d['reason']} {d['size_px']:.0f}px cls={d['cls']}")
plt.tight_layout(); plt.show()

## 5. Train the head -- trunk FROZEN

`FINETUNE = False` is the deliverable. Flipping it to True unfreezes the trunk at
lr/10 under a joint loss (BCE + error head + corner smooth-L1) -- do that ONLY if the
frozen run fails the frozen criteria, and then the cross-eval cell at the bottom must
clear the finetuned regressor on the clean subset before the pair can ship; a finetuned
trunk with a regressed clipped row is a worse aircraft, whatever its ROC looks like.

In [ ]:
import types, threading

TAG       = "colab-conf"
EPOCHS    = 60
BATCH     = 256
WORKERS   = min(8, os.cpu_count() or 2)
MAX_HOURS = 2.5
RESUME    = False        # flip after a disconnect and Run all again
FINETUNE  = False        # see the cell above before touching this

# LR SCALES WITH BATCH: the local default is 1e-3 at batch 64 (head-only, AdamW).
# Holding 1e-3 while quadrupling the batch quarters the per-sample step -- the run
# would look worse purely from schedule. Linear scaling, same as the regressor runs.
LR = 1e-3 * (BATCH / 64)

RUNS_LOCAL = "/content/gatenet_runs"
os.makedirs(f"{RUNS_LOCAL}/{TAG}", exist_ok=True)
C.RUNS = RUNS_LOCAL

_stop = threading.Event()
def _mirror(every=180):
    while not _stop.wait(every):
        try:
            for n in ("best.pt", "last.pt", "log.csv"):
                s = f"{RUNS_LOCAL}/{TAG}/{n}"
                if os.path.exists(s):
                    shutil.copyfile(s, f"{RUNS_DRIVE}/{TAG}/{n}")
        except Exception as e:
            print("mirror failed (training continues):", e)
threading.Thread(target=_mirror, daemon=True).start()

if RESUME:
    for n in ("best.pt", "last.pt", "log.csv"):
        s = f"{RUNS_DRIVE}/{TAG}/{n}"
        if os.path.exists(s) and not os.path.exists(f"{RUNS_LOCAL}/{TAG}/{n}"):
            shutil.copyfile(s, f"{RUNS_LOCAL}/{TAG}/{n}")

args = types.SimpleNamespace(
    mode="train", tag=TAG, trunk=TRUNK_LOCAL,
    labels=G.LABELS, epochs=EPOCHS, batch=BATCH, lr=LR, workers=WORKERS,
    max_hours=MAX_HOURS, resume=RESUME, finetune=FINETUNE, cpu=False, seed=0,
    limit=0, allow_unmined=False)

torch.manual_seed(0); np.random.seed(0); cv2.setNumThreads(0)
t0 = time.time()
C.train(args)
_stop.set()
for n in ("best.pt", "last.pt", "log.csv"):
    s = f"{RUNS_LOCAL}/{TAG}/{n}"
    if os.path.exists(s):
        shutil.copyfile(s, f"{RUNS_DRIVE}/{TAG}/{n}")
print(f"wall clock {(time.time()-t0)/60:.1f} min; checkpoints on Drive: {RUNS_DRIVE}/{TAG}")

## 6. The decision tables

Clean-eval discipline (the leakage section of TRAINING.md): true-gate loss and
catastrophe rows use ONLY positives that are honest for the frozen regressor --
VQ1 frames in `val_clean_keys.json` (held out of BOTH regressor splits) plus VQ2 hand
instances. Deco rejection uses the conf-val mined decoration crops the head never saw.

In [ ]:
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ck = torch.load(f"{RUNS_LOCAL}/{TAG}/best.pt", map_location=dev, weights_only=False)
trunk = C.load_trunk(TRUNK_LOCAL, dev)
if ck.get("finetune") and "trunk" in ck:
    trunk.load_state_dict(ck["trunk"])
    print("NOTE: finetuned trunk loaded -- the cross-eval cell below is now a GATE")
head = C.ConfHead().to(dev)
head.load_state_dict(ck["head"])

_clean = set(json.load(open(f"{CODE_LOCAL}/pilot/perception/val_clean_keys.json"))["clean"])
aux = [d for d in items if d["cls"] == 1 and (d["key"] in _clean or d["src"] == "hand")]
report, r = C.eval_report(trunk, head, va_items, dev, workers=WORKERS,
                          clean_keys=f"{CODE_LOCAL}/pilot/perception/val_clean_keys.json",
                          aux_items=aux)
with open(f"{RUNS_DRIVE}/{TAG}/eval_report.md", "w") as fh:
    fh.write(report)
print(f"\nreport mirrored to {RUNS_DRIVE}/{TAG}/eval_report.md")

## 7. Regressor integrity

Frozen run: the corners with the head attached must be BIT-identical to the production
trunk alone -- an equality assert, not a tolerance. Finetuned run: the finetuned
regressor is cross-evaled against production on the clean-key positives; it ships only
if it does not regress (ALL and clipped rows), same rule as every regressor retrain.

In [ ]:
xb = torch.stack([C.ConfCrops(va_items, False)[k][0]
                  for k in np.linspace(0, len(va_items) - 1, 16).astype(int)]).to(dev)
prod = C.load_trunk(TRUNK_LOCAL, dev)
with torch.no_grad():
    a = prod(xb)
    b, _, _ = C.conf_forward(trunk, head, xb)

if not ck.get("finetune"):
    assert torch.equal(a, b), "regressor output moved with the head attached!"
    print("regressor path BIT-IDENTICAL with head attached -- production corners untouched")
else:
    from torch.utils.data import DataLoader
    clean = set(json.load(open(f"{CODE_LOCAL}/pilot/perception/val_clean_keys.json"))["clean"])
    pos = [d for d in va_items if d["cls"] == 1 and (d["key"] in clean or d["src"] == "hand")]
    gitems = [{'path': d['path'], 'session': d['session'], 'ordinal': 0,
               'corners': d['gt'], 'clipped': d['clipped'], 'occluded': False,
               'size_px': d['size_px'], 'src': d['src'], 'body_rate': 0.0} for d in pos]
    ld = DataLoader(G.GateCrops(gitems, False), batch_size=256, num_workers=WORKERS)
    for name, m in (("production", prod), ("finetuned", trunk)):
        res = G.evaluate(m, ld, dev, gitems)
        rows = G.breakdown(res, gitems)
        byg = {r_["group"]: r_ for r_ in rows}
        print(f"{name:10s} ALL {byg['ALL']['corner_med']:.2f}/{byg['ALL']['corner_p90']:.2f}  "
              f"clipped {byg.get('clipped',{}).get('corner_med',float('nan')):.2f}")
    print("ship the PAIR only if finetuned does not regress ALL or clipped above.")

## 8. Latency (Colab CPU is indicative; the laptop bench already measured 0.21 ms)

In [ ]:
bench_args = types.SimpleNamespace(trunk=TRUNK_LOCAL)
C.bench(bench_args)

## 9. What is on Drive

In [ ]:
print("on Drive:", f"{RUNS_DRIVE}/{TAG}")
for f in sorted(os.listdir(f"{RUNS_DRIVE}/{TAG}")):
    print(f"  {f}  {os.path.getsize(f'{RUNS_DRIVE}/{TAG}/{f}')/1e6:.2f} MB")
print("\nBring home: copy gatenet_runs/colab-conf/ into pilot/perception/gatenet_runs/,")
print("then locally:  python3 pilot/perception/gatenet_conf.py --mode eval --tag colab-conf")
print("and paste eval_report.md + the ACCEPT/REJECT line into TRAINING.md.")